In [1]:
# -*- coding: utf-8 -*-
"""
Detecção Automatizada de Embarcações de Garimpo Ilegal - INFERÊNCIA RÁPIDA
Autor: Fellipe Machado Castro (Universidade Federal do Pará - UFPA)
"""
print("Instalando bibliotecas necessárias...")
!pip install -q rasterio geopandas segmentation-models-pytorch

import os, glob
import numpy as np
import rasterio
import geopandas as gpd
from rasterio.windows import Window
from shapely.geometry import box
import torch
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy.ndimage import label, find_objects, binary_opening
from scipy.signal.windows import gaussian as gaussian_window
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

Instalando bibliotecas necessárias...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.9 MB/s eta 0:00:00
Mounted at /content/drive


In [2]:
# --- CONFIGURAÇÕES ---
BASE_DIR = "/content/drive/MyDrive/Dados_TCC_Garimpo"
RESULT_DIR = os.path.join(BASE_DIR, "resultados_inferencia")
os.makedirs(RESULT_DIR, exist_ok=True)

ENCODER_NAME = "resnet34"
MODEL_SAVE_PATH = os.path.join(BASE_DIR, f"unet_{ENCODER_NAME}.pth")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Parâmetros descobertos no TCC (Evita recálculos pesados)
BEST_THR = 0.40  # Limiar otimizado do F1-Score
CHIP_SIZE = 256
STRIDE = 128

print(f"Dispositivo: {DEVICE.upper()}")
print("Carregando arquitetura e pesos do modelo treinado...")

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=None,
    in_channels=1,
    classes=1
).to(DEVICE)

model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

print("Modelo pronto para uso!")

Dispositivo: CUDA
Carregando arquitetura e pesos do modelo treinado...
Modelo pronto para uso!


In [3]:
# --- FUNÇÕES DE APOIO ---
def normalize_sar(img):
    img = img.astype(np.float32)
    img = np.where(img < -999, np.nan, img)
    img = np.nan_to_num(img, nan=np.nan)
    finite = img[np.isfinite(img)]
    if finite.size == 0: return np.zeros_like(img, dtype=np.float32)
    lo, hi = np.percentile(finite, 2), np.percentile(finite, 99)
    img = np.nan_to_num(img, nan=lo)
    img = np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)
    return img.astype(np.float32)

def predict_tta(model, x):
    outs = []
    for k in range(4):
        xx = torch.rot90(x, k, dims=[2,3])
        with torch.no_grad(): p = torch.sigmoid(model(xx))
        outs.append(torch.rot90(p, -k, dims=[2,3]))
    with torch.no_grad(): p = torch.sigmoid(model(torch.flip(x, dims=[3])))
    outs.append(torch.flip(p, dims=[3]))
    return torch.stack(outs).mean(0)

# --- RECONSTRUÇÃO DA CENA ---
TIF_FILES = glob.glob(os.path.join(BASE_DIR, "imagens_sar_originais", "S1_*.tif"))
_g1d = gaussian_window(CHIP_SIZE, std=CHIP_SIZE/4)
GAUSS_KERNEL = np.outer(_g1d, _g1d).astype(np.float32)
GAUSS_KERNEL /= GAUSS_KERNEL.max()

print(f"Iniciando varredura em {len(TIF_FILES)} imagens de satélite...")

for tif_file in TIF_FILES:
    fname = os.path.basename(tif_file)
    with rasterio.open(tif_file) as src:
        H, W = src.height, src.width
        full_img = src.read(1)
        prob_map   = np.zeros((H,W), dtype=np.float32)
        weight_map = np.zeros((H,W), dtype=np.float32)

        for y in range(0, H, STRIDE):
            for x in range(0, W, STRIDE):
                window = Window(x, y, CHIP_SIZE, CHIP_SIZE).intersection(Window(0,0,W,H))
                chip = src.read(1, window=window)
                if chip.size == 0: continue
                hh, ww = chip.shape
                pad = np.zeros((CHIP_SIZE, CHIP_SIZE), dtype=np.float32)
                pad[:hh,:ww] = chip
                norm = normalize_sar(pad)
                pred = predict_tta(model, torch.from_numpy(norm).unsqueeze(0).unsqueeze(0).to(DEVICE)).cpu().numpy()[0,0]
                ker  = GAUSS_KERNEL[:hh,:ww]
                prob_map[y:y+hh, x:x+ww]   += pred[:hh,:ww]*ker
                weight_map[y:y+hh, x:x+ww] += ker

        prob_map /= (weight_map + 1e-6)
        binary_mask = binary_opening(prob_map > BEST_THR, iterations=1) # Usando o BEST_THR de 0.40
        labeled, n = label(binary_mask)
        objs = find_objects(labeled)
        valid = [o for o in objs if (o[0].stop-o[0].start)*(o[1].stop-o[1].start) >= 12]

        print(f"{fname} | Embarcações Detectadas: {len(valid)}")

        plt.figure(figsize=(15,15))
        plt.imshow(normalize_sar(full_img), cmap='gray'); ax = plt.gca()
        for o in valid:
            y0,y1 = o[0].start, o[0].stop; x0,x1 = o[1].start, o[1].stop
            ax.add_patch(patches.Rectangle((x0,y0), x1-x0, y1-y0, linewidth=2, edgecolor='red', facecolor='none'))
        plt.title(f"Detecções na Cena Inteira | {fname}", fontsize=14)
        plt.axis('off')
        plt.savefig(os.path.join(RESULT_DIR, f"INFERENCIA_{fname}.png"), bbox_inches='tight', dpi=150); plt.close()

print("Varredura concluída! Imagens geradas na pasta resultados_inferencia.")

Iniciando varredura em 15 imagens de satélite...
S1_SaoCarlos_2025_8.tif | Embarcações Detectadas: 0
S1_JaciParana_2025_6.tif | Embarcações Detectadas: 12
S1_ControleHumaita_2025_8.tif | Embarcações Detectadas: 0
S1_SaoCarlos_2025_6.tif | Embarcações Detectadas: 1
S1_ControleHumaita_2025_6.tif | Embarcações Detectadas: 0
S1_PortoVelho_2025_6.tif | Embarcações Detectadas: 19
S1_SaoCarlos_2025_7.tif | Embarcações Detectadas: 0
S1_ControleHumaita_2025_7.tif | Embarcações Detectadas: 0
S1_JaciParana_2025_7.tif | Embarcações Detectadas: 11
S1_JaciParana_2025_8.tif | Embarcações Detectadas: 13
S1_PortoVelho_2025_7.tif | Embarcações Detectadas: 20
S1_PortoVelho_2025_8.tif | Embarcações Detectadas: 18
S1_BoaHora_2025_7.tif | Embarcações Detectadas: 9
S1_BoaHora_2025_6.tif | Embarcações Detectadas: 1
S1_BoaHora_2025_8.tif | Embarcações Detectadas: 5
Varredura concluída! Imagens geradas na pasta resultados_inferencia.
